# Lab 6 — Linear Regression with PyTorch

**Student:** Hitesh Gupta | **Roll No.:** IC-2k23-39

Implements a linear regression model `y = wx + b` using PyTorch with
a custom training loop, MSE loss, and SGD/Adam optimisers.
Application: student score / house price prediction.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

## 1. Generate Synthetic Dataset

In [ ]:
# True relationship: y = 2.5x + 1.0 + noise
n = 100
x_np = np.random.uniform(0, 10, n)
y_np = 2.5 * x_np + 1.0 + np.random.normal(0, 1.5, n)

X = torch.tensor(x_np, dtype=torch.float32).unsqueeze(1)
Y = torch.tensor(y_np, dtype=torch.float32).unsqueeze(1)

plt.scatter(x_np, y_np, alpha=0.6, label='Data')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Synthetic Linear Regression Data')
plt.legend()
plt.savefig('outputs/data.png', dpi=300, bbox_inches='tight')
plt.show()

## 2. Define Model

In [ ]:
class LinearRegressionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(1, 1)
    def forward(self, x):
        return self.linear(x)

model = LinearRegressionModel()
print(model)
print('Initial weights:', model.linear.weight.item(),
      'bias:', model.linear.bias.item())

## 3. Training Loop (SGD vs Adam)

In [ ]:
def train(optimizer_name='sgd', lr=0.01, epochs=500):
    torch.manual_seed(42)
    model = LinearRegressionModel()
    criterion = nn.MSELoss()
    if optimizer_name == 'sgd':
        opt = torch.optim.SGD(model.parameters(), lr=lr)
    else:
        opt = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []
    for epoch in range(epochs):
        opt.zero_grad()
        pred = model(X)
        loss = criterion(pred, Y)
        loss.backward()
        opt.step()
        losses.append(loss.item())
    return model, losses

model_sgd, loss_sgd = train('sgd', lr=0.05, epochs=500)
model_adam, loss_adam = train('adam', lr=0.05, epochs=500)

print('SGD  final loss:', loss_sgd[-1])
print('Adam final loss:', loss_adam[-1])
print('SGD  weights:', model_sgd.linear.weight.item(),
      'bias:', model_sgd.linear.bias.item())
print('Adam weights:', model_adam.linear.weight.item(),
      'bias:', model_adam.linear.bias.item())

In [ ]:
# Loss curves
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(loss_sgd, label='SGD')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('SGD Convergence')
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(loss_adam, label='Adam', color='orange')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Adam Convergence')
plt.legend()
plt.tight_layout()
plt.savefig('outputs/loss_curves.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Prediction & Evaluation

In [ ]:
with torch.no_grad():
    pred_sgd = model_sgd(X)
    pred_adam = model_adam(X)

mse_sgd = nn.MSELoss()(pred_sgd, Y).item()
mse_adam = nn.MSELoss()(pred_adam, Y).item()
print('Test MSE (SGD) :', round(mse_sgd, 4))
print('Test MSE (Adam):', round(mse_adam, 4))

# Plot fitted lines
plt.scatter(x_np, y_np, alpha=0.5, label='Data')
x_line = torch.linspace(0, 10, 100).unsqueeze(1)
with torch.no_grad():
    y_sgd = model_sgd(x_line).numpy()
    y_adam = model_adam(x_line).numpy()
plt.plot(x_line.numpy(), y_sgd, 'r-', label='SGD')
plt.plot(x_line.numpy(), y_adam, 'g--', label='Adam')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Fitted Linear Regression Lines')
plt.legend()
plt.savefig('outputs/fitted_lines.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Save Model

In [ ]:
import os
os.makedirs('../models', exist_ok=True)
torch.save(model_adam.state_dict(), '../models/linear_regression.pt')
print('Model saved to ../models/linear_regression.pt')

# Verify reload
model_loaded = LinearRegressionModel()
model_loaded.load_state_dict(torch.load('../models/linear_regression.pt'))
print('Reloaded weights:', model_loaded.linear.weight.item(),
      'bias:', model_loaded.linear.bias.item())

## Summary

- PyTorch tensors support NumPy-like operations with autograd.
- `nn.Linear(1,1)` implements `y = wx + b`.
- MSE loss + SGD/Adam converge to weights close to true values (w=2.5, b=1.0).
- Adam typically converges faster than SGD for this problem.
- Model saved via `torch.save` / `state_dict` for reuse.